In [5]:
%pip install pillow kagglehub pytorch-lightning


Note: you may need to restart the kernel to use updated packages.


In [6]:
import os

import kagglehub
import pytorch_lightning as PTL
import torch
import torch.optim as optim
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms


In [7]:
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("path", path)


100%|██████████| 818M/818M [03:36<00:00, 3.95MB/s] 

Extracting files...


path /Users/imac/.cache/kagglehub/datasets/mohitsingh1804/plantvillage/versions/1


train + validation +test

In [9]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device)

device cpu


In [21]:
TRAIN_PATH = os.path.join(path, "plantvillage", "train")
VAL_PATH = os.path.join(path, "plantvillage", "test")

In [11]:
transform=transforms.Compose(
    [transforms.Resize((128, 128)),
     transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

custom dataset 

In [18]:
from genericpath import isfile

class MultiClassClassification(Dataset):
    def __init__(self,root_dir,transform=None):
        super().__init__()
        self.samples = []
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            for fname in os.listdir(cls_dir):
                fpath = os.path.join(cls_dir, fname)
                if isfile(fpath):
                    label=self.class_to_idx[cls_name]
                    self.samples.append((fpath, self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.samples)
    def __getitem__(self, index):
        fpath, label = self.samples[index]
        img = Image.open(fpath).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

In [17]:
students = ["Alice", "Bob", "Charlie", "David", "Eve"]
scores = [85, 92, 78, 90, 88]
for idx ,student in enumerate(students):
    print(f"{idx}: {student} scored {scores[idx]}")

0: Alice scored 85
1: Bob scored 92
2: Charlie scored 78
3: David scored 90
4: Eve scored 88


Load data 

In [19]:
import os 
print(os.listdir(path))

['PlantVillage']


In [20]:
train_dataset = MultiClassClassification(TRAIN_PATH, transform=transform)
val_dataset = MultiClassClassification(VAL_PATH, transform=transform)
num_classes = len(train_dataset.classes)
print("num_classes", num_classes)
print("train_dataset", len(train_dataset))
print("val_dataset", len(val_dataset))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/imac/.cache/kagglehub/datasets/mohitsingh1804/plantvillage/versions/1/plantvillage/test'